# Homework 5 RF Accuracy Improvement

In [9]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_squared_error
import joblib

# Load the weather data
weather = pd.read_csv(r'C:\Users\Campb\OneDrive\Documents\CSCI_4120\Decision_tree\data\BicycleWeather.csv')  
weather['DATE'] = pd.to_datetime(weather['DATE'], format='%Y%m%d')

# Load the Fremont Bridge traffic data
daily = pd.read_csv(r'C:\Users\Campb\OneDrive\Documents\CSCI_4120\Decision_tree\data\FremontBridge.csv')  
daily['DATE'] = pd.to_datetime(daily['DATE'], format='%m/%d/%Y %I:%M:%S %p')

# Resample traffic data to hourly
daily.set_index('DATE', inplace=True)
daily_hourly = daily.resample('h').ffill()

# Resample the weather data to hourly
weather.set_index('DATE', inplace=True)
weather_hourly = weather.resample('h').ffill()

# Merge the traffic data with the resampled weather data
merged_data = pd.merge(daily_hourly, weather_hourly[['PRCP', 'TMAX', 'TMIN', 'AWND']], on='DATE', how='left')

# Handle missing values (Option 1: Drop NaN values)
merged_data_clean = merged_data.dropna(subset=['PRCP', 'TMAX', 'TMIN', 'AWND', 'Fremont Bridge East Sidewalk'])

# Prepare feature matrix (X) and target vector (y)
X = merged_data_clean[['PRCP', 'TMAX', 'TMIN', 'AWND']]
y = merged_data_clean['Fremont Bridge East Sidewalk']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define models
models = {
    'Linear Regression': LinearRegression(),
    'Lasso': Lasso(),
    'Ridge': Ridge()
}

# Lasso and Ridge alpha values range for RandomizedSearchCV
param_distributions = {
    'Lasso': {'alpha': np.logspace(-4, 4, 20)},
    'Ridge': {'alpha': np.logspace(-4, 4, 20)}
}

# Initialize best models dictionary
best_models = {}

# Tune and evaluate models
for model_name, model in models.items():
    print(f'Tuning {model_name}...')
    
    if model_name in ['Lasso', 'Ridge']:
        # Apply RandomizedSearchCV with adjusted n_iter
        random_search = RandomizedSearchCV(model, param_distributions[model_name], n_iter=20, cv=10, random_state=42)
        random_search.fit(X_train, y_train)
        
        # Save best model
        best_models[model_name] = random_search.best_estimator_
        
        print(f'Best alpha for {model_name}: {random_search.best_params_["alpha"]}')
        print(f'Best cross-validation score: {random_search.best_score_:.4f}')
    else:
        # For Linear Regression, no hyperparameter tuning needed
        model.fit(X_train, y_train)
        best_models[model_name] = model
        print(f'{model_name} training score: {model.score(X_train, y_train):.4f}')

# Evaluate the models on the test set
for model_name, model in best_models.items():
    print(f'\nEvaluating {model_name}...')
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2 = model.score(X_test, y_test)
    print(f'Mean Squared Error: {mse:.4f}')
    print(f'R²: {r2:.4f}')

# Find the best model based on R² score
best_model_name = max(best_models, key=lambda model_name: best_models[model_name].score(X_test, y_test))
print(f'\nBest model: {best_model_name}')

# Save model info to text file
best_model = best_models[best_model_name]
model_info = f"Model: {best_model_name}\n"

# Save coefficients and intercept for Linear Regression, Lasso, and Ridge models
if isinstance(best_model, LinearRegression):
    model_info += f"Coefficients: {best_model.coef_}\n"
    model_info += f"Intercept: {best_model.intercept_}\n"
elif isinstance(best_model, (Lasso, Ridge)):
    model_info += f"Coefficients: {best_model.coef_}\n"
    model_info += f"Intercept: {best_model.intercept_}\n"
    # Save the best alpha value for Lasso and Ridge
    if isinstance(best_model, Lasso):
        model_info += f"Best Alpha: {random_search.best_params_['alpha']}\n"
    elif isinstance(best_model, Ridge):
        model_info += f"Best Alpha: {random_search.best_params_['alpha']}\n"

# Save to text file
with open("best_model_info.txt", "w") as f:
    f.write(model_info)


Tuning Linear Regression...
Linear Regression training score: 0.0688
Tuning Lasso...
Best alpha for Lasso: 0.23357214690901212
Best cross-validation score: 0.0680
Tuning Ridge...
Best alpha for Ridge: 10000.0
Best cross-validation score: 0.0680

Evaluating Linear Regression...
Mean Squared Error: 5248.2257
R²: -0.0053

Evaluating Lasso...
Mean Squared Error: 5242.1844
R²: -0.0041

Evaluating Ridge...
Mean Squared Error: 5246.5127
R²: -0.0049

Best model: Lasso
